# 13 · Circular reasoning & training-data leakage

> **The question this notebook answers:** *How do you avoid circular reasoning
> when you say "predictor X disagrees with ClinVar"? Did you account for each
> tool's training-data cut-off?*

**Honest answer for the A1/A2 work as it stands: no.** The original discordance
worklist compared predictors to ClinVar without (a) checking whether a predictor
was *trained on* ClinVar-lineage labels, or (b) restricting the comparison to
ClinVar records newer than each tool's training freeze. This notebook explains
the problem and shows the safer way to do it.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
%matplotlib inline

## 1. Two kinds of predictor — and why it decides everything

A variant-effect predictor is only independent evidence *if it did not learn from
the answer key you are testing it against.*

- **Unsupervised** (AlphaMissense, EVE, ESM1b): trained on **protein sequences /
  evolutionary alignments only** — never on clinical pathogenic/benign labels.
  Comparing them to ClinVar is (largely) fair.
- **Semi-supervised** (PrimateAI, CADD): trained on a *proxy* (common/primate
  variants = "probably benign", simulated variants = "probably not observed).
  Some leakage risk.
- **Supervised** (REVEL): trained **directly on curated pathogenic/benign
  variants** whose lineage overlaps ClinVar/HGMD. Comparing REVEL to ClinVar is
  **partly circular** — high agreement is partly memorisation, and a
  "disagreement" may just be label noise, not new evidence.

In [2]:
reg = (pd.DataFrame(tk.TOOL_REGISTRY).T
         .reset_index().rename(columns={"index":"tool"})
         [["tool","kind","learning","circularity","signal"]])
# order from safest to most circular for ClinVar benchmarking
order = {"low":0,"low-for-clinvar":0,"medium":1,"HIGH (label lineage overlaps ClinVar/HGMD)":2}
reg["rank"] = reg["circularity"].map(lambda c: order.get(c, 1))
reg.sort_values("rank")[["tool","kind","learning","circularity"]].reset_index(drop=True)

,tool,kind,learning,circularity
0,AlphaMissense,missense,unsupervised,low
1,EVE,missense,unsupervised,low
2,ESM1b,missense,unsupervised,low
3,SpliceAI,splice,"supervised (on GTEx splice junctions, not clin...",low-for-clinvar
4,Pangolin,splice,supervised (on splice usage across species/tis...,low-for-clinvar
5,PrimateAI,missense,semi-supervised,medium
6,CADD,general,semi-supervised,medium
7,REVEL,missense,supervised,HIGH (label lineage overlaps ClinVar/HGMD)


## 2. The circularity mechanism, concretely

Suppose REVEL was trained on a set of variants labelled *Pathogenic* that later
became ClinVar *Pathogenic* entries. Then:

- When you test "does REVEL agree with ClinVar?", the variants REVEL **memorised**
  inflate the agreement — the benchmark looks better than the tool really is.
- When REVEL **disagrees** with ClinVar on a VUS, you cannot tell whether that is
  *independent computational evidence for reclassification* (what you want) or
  simply *a variant REVEL never learned well* (label noise).

This is why the A1 "413 discordances" are safest when they come from an
**unsupervised** tool. And in fact they do — the 413 list is
**AlphaMissense-vs-ClinVar** (notebook 12), and AlphaMissense is unsupervised.
The genuinely dangerous move would be to build the same worklist from **REVEL**
vs ClinVar and treat it as independent.

## 3. Training-data cut-off (temporal leakage)

Even an unsupervised tool is calibrated at a point in time, and supervised tools
freeze their training labels on some date. **Proper evaluation uses only ClinVar
records that were submitted *after* the tool's training freeze** — otherwise you
are testing on data the tool may have seen.

The A1 pipeline did **not** record or use these dates. To do it right you would:

1. Find each tool's training-data freeze (from its paper / release notes).
2. Pull ClinVar's `LastEvaluated` date per variant (it is in the raw
   `variant_summary` — the toolkit keeps `review_status`; the date column is
   available in the same file).
3. Keep only variants **first classified after** the tool's freeze → a *temporally
   held-out* test set.

Approximate freezes (⚠ **verify from each paper before using — these are rough**):

| Tool | ~Training freeze | Trained on clinical labels? |
|---|---|---|
| AlphaMissense | 2023 | No (sequence/structure) |
| EVE | 2021 | No (MSA) |
| ESM1b | 2022 | No (UniRef sequences) |
| PrimateAI | 2018 | Proxy only (primate/common) |
| REVEL | 2016 | **Yes** (HGMD + ESP) |
| SpliceAI | 2019 | No (GTEx junctions) |
| CADD | per release | Proxy (simulated vs observed) |

The key column here is the last one, not the date: a tool that never saw clinical
labels cannot leak them regardless of date.

## 4. The safer benchmark — use *orthogonal* truth, not ClinVar

The cleanest fix is to judge sequence predictors against evidence of a **different
kind**: an experimental **functional assay**. For CFTR that is **CFTR2**, whose
calls fold in *in-vitro* CFTR function (chloride conductance / processing), and
the **Raraigh 2018** functional dataset. A predictor agreeing with a wet-lab
functional readout is far stronger than agreeing with another in-silico-adjacent
label.

Below: a tiny illustration using the DEMO CFTR2 functional classes as the truth
set, scoring the **unsupervised** tools against it (demo data — method only).

In [3]:
demo = tk._demo_frame().copy()
# truth = CFTR2 functional class collapsed to path/benign
def cftr2_truth(c):
    c = str(c)
    if "CF-causing" in c: return "pathogenic"
    if "Not CF" in c:     return "benign"
    return None                      # VUS / unknown -> excluded from a benchmark
demo["truth"] = demo["cftr2_class"].apply(cftr2_truth)

for tool, col in [("am","am_score"),("eve","eve_score"),("esm1b","esm1b_score")]:
    demo[tool+"_call"] = demo[col].apply(lambda s: tk.call_from_score(s, tool))

bench = demo.dropna(subset=["truth"])[
    ["protein_variant","cftr2_class","truth","am_call","eve_call","esm1b_call"]]
bench.reset_index(drop=True)

,protein_variant,cftr2_class,truth,am_call,eve_call,esm1b_call
0,G551D,CF-causing,pathogenic,pathogenic,pathogenic,pathogenic
1,F508del,CF-causing,pathogenic,na,na,na
2,R117H,CF-causing (mild),pathogenic,uncertain,benign,benign
3,R334W,CF-causing,pathogenic,pathogenic,pathogenic,pathogenic
4,G85E,CF-causing,pathogenic,pathogenic,pathogenic,pathogenic
5,R668C,Not CF-causing,pathogenic,benign,benign,benign
6,M470V,Not CF-causing,pathogenic,benign,benign,benign


In [4]:
# concordance of each unsupervised tool vs the CFTR2 functional truth (demo)
def concordance(call_col):
    ok = ((bench["truth"]=="pathogenic") & (bench[call_col]=="pathogenic")) | \
         ((bench["truth"]=="benign")     & (bench[call_col]=="benign"))
    scored = bench[call_col].isin(["pathogenic","benign"])
    return f"{ok.sum()}/{scored.sum()} concordant (of {len(bench)} truth variants)"
for t in ["am_call","eve_call","esm1b_call"]:
    print(f"{t:12s}: {concordance(t)}")
print("\n(DEMO data — this is the METHOD, not a real benchmark result.)")

am_call     : 3/5 concordant (of 7 truth variants)
eve_call    : 3/6 concordant (of 7 truth variants)
esm1b_call  : 3/6 concordant (of 7 truth variants)

(DEMO data — this is the METHOD, not a real benchmark result.)


## 5. A de-circularization checklist

Use this before claiming a predictor "disagrees with the clinical call":

1. **Is the predictor supervised on clinical labels?** If yes (REVEL), do not treat
   its ClinVar disagreement as independent — or down-weight it.
2. **Prefer unsupervised tools** (AlphaMissense / EVE / ESM1b) for ClinVar-based
   discordance worklists.
3. **Temporal hold-out:** restrict to ClinVar variants classified *after* the
   tool's training freeze.
4. **Orthogonal truth:** validate against *functional* evidence (CFTR2,
   Raraigh 2018 assays), not another in-silico-adjacent label.
5. **Mind ClinVar review status** (notebook 07): weight expert-panel (3-star)
   assertions above single-submitter VUS.
6. **Report tool independence:** an ensemble of predictors that *all* trained on
   overlapping data is not five independent votes (this is exactly the trap in the
   "≥3/5 tools" rule — REVEL itself already contains 13 predictors).

## Key takeaways

- "Predictor disagrees with ClinVar" is only *evidence* if the predictor never
  learned from ClinVar-lineage labels. **REVEL is the one to distrust here;
  AlphaMissense/EVE/ESM1b are the safe ones.**
- The A1 worklist happens to be built from **unsupervised AlphaMissense**, so its
  core is defensible — but it did **not** apply a temporal hold-out or record
  training freezes, and it should be validated against **CFTR2 functional** truth
  before any reclassification.
- The "≥3/5 tools" consensus is **not five independent votes** — the tools share
  training data and REVEL is itself an ensemble of 13. Treat it as weak
  corroboration, not five witnesses.